# 00. Context Window 与 Token 预算

Context window 是 Agent 最稀缺的资源。每往 prompt 里放 1K token，你同时支付三份成本：
- **钱**：每个 turn 都为全部前缀付费
- **延迟**：首 token 时间随 prompt 长度增长
- **质量**：「Lost in the Middle」— 长上下文中段信息利用率显著下降

> **检查点**：能解释为什么 tool schema 的 token 开销要按「每轮 × 对话轮数」计算总成本。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()

🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'context-engineering' / 'scripts' / 'context_budget_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

Component                      Tokens  Δ vs prev  Mode
------------------------------------------------------------
system + query                    115       +115  heuristic
+ 4 few-shot examples             273       +158  heuristic
+ 3 tool schemas                  493       +220  heuristic

--- Component sizes (chars) ---
system prompt : 400
few-shot total: 631
tool schemas  : 880
user query    : 61

Takeaway: tool schemas and few-shot examples often cost more than the system prompt.
Every token is paid on EVERY turn — budget them like memory, not like disk.



## 预算分配经验法则

| 组件 | 典型占比 | 优化手段 |
|------|---------|---------|
| System prompt | 5-15% | 精简规则，删除从未触发的条款 |
| Few-shot 示例 | 10-30% | 减少到 2-3 个；用 RAG 按需注入 |
| Tool schemas | 10-40% | 裁剪 description；懒加载工具（先给目录，选中后再给完整 schema） |
| 对话历史 | 增长无界 | Compaction（下一个 notebook） |

> **检查点**：如果一个 agent 有 20 个工具，每个 schema 200 token，10 轮对话光 schema 就花多少 token？（答案：20×200×10 = 40K）

In [3]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'context-engineering' / 'scripts' / 'context_budget_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--use-api', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

Component                      Tokens  Δ vs prev  Mode
------------------------------------------------------------
system + query                    111       +111  API usage
+ 4 few-shot examples             301       +190  API usage
+ 3 tool schemas                  733       +432  API usage

--- Component sizes (chars) ---
system prompt : 400
few-shot total: 631
tool schemas  : 880
user query    : 61

Takeaway: tool schemas and few-shot examples often cost more than the system prompt.
Every token is paid on EVERY turn — budget them like memory, not like disk.



## 真实测量观察

对比上面两次运行：heuristic（4 chars/token）与真实 usage 的差值。中文场景的
chars/token 比率与英文不同 — 这就是为什么生产系统必须用 usage 字段而非估算。

## 练习

1. 修改 `context_budget_demo.py` 的 `TOOL_SCHEMAS`，把工具数量加到 10 个，观察 token 增长曲线。
2. 把 few-shot 示例从 4 个减到 2 个，量化节省的 token；回答质量受多大影响？（用 `--use-api` 实际问一个问题对比）
3. 思考：什么信息应该放在 system prompt，什么应该按需检索注入？划分的依据是什么？